# 🔥 MapBiomas Fire — Sentinel-2 Monitor
## Piloto Perú · Ejecución Local

Versión para ejecutar localmente (conda / venv).  
Misma lógica que el notebook de Colab, pero con autenticación vía `gcloud` y rutas locales.

### Prerrequisitos
```bash
conda create -n fire_monitor python=3.10
conda activate fire_monitor
pip install earthengine-api gcsfs rasterio scipy tensorflow==2.13.0 ipywidgets
conda install -c conda-forge gdal   # o: brew install gdal / apt install gdal-bin
gcloud auth application-default login
earthengine authenticate
```

## ⚙️ M0 — Configuración y Autenticación Local

In [ ]:
import sys, os

# ── Apuntar a los módulos locales
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))  # ajustar según sea necesario
SRC_PATH  = os.path.join(REPO_ROOT, 'src', 'colab')

if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f'✅ Módulos en: {SRC_PATH}')

In [ ]:
# ── Autenticación
# Requiere: gcloud auth application-default login (ADC para GCS)
#         earthengine authenticate               (GEE)

from M0_auth_config import CONFIG, authenticate, print_config
authenticate()
print_config()

In [ ]:
# ── Sobrescribir configuraciones si es necesario (p. ej. carpeta local para descargas)
# CONFIG['bucket'] = 'mi-bucket-alternativo'
# CONFIG['ee_project'] = 'mi-proyecto-ee'
print('Configuración activa:', CONFIG['country'], '|', CONFIG['ee_project'])

## 🛰️ M1 — Generador de Mosaicos

In [ ]:
# ── Interfaz de envío (funciona vía jupyter lab/notebook localmente con ipywidgets)
from M1_mosaic_generator import run_ui as mosaic_ui
mosaic_interface = mosaic_ui()

In [ ]:
# ── Uso programático (sin interfaz de usuario) — útil para la automatización local
# from M1_mosaic_generator import build_mosaic, export_to_asset, export_to_gcs_chunks
# from M1_mosaic_generator import assemble_country_mosaic, check_mosaic_status, generate_grid
# import ee
#
# year, month = 2024, 8
#
# # 1. Verificar estado
# status = check_mosaic_status(year, months=[month], period='monthly')
# print(status)
#
# # 2. Enviar mosaico
# from M0_auth_config import get_country_geometry, mosaic_name
# geom   = get_country_geometry()
# start  = ee.Date(f'{year}-{month:02d}-01')
# end    = start.advance(1, 'month')
# mosaic = build_mosaic(start, end, geom, apply_focus_mask=True, year=year, month=month)
# name   = mosaic_name(year, month, 'monthly')
# tiles  = generate_grid(geom)
# t_asset  = export_to_asset(mosaic, name, year, month, 'monthly')
# t_chunks = export_to_gcs_chunks(mosaic, name, year, month, 'monthly', tiles)
# print(f'Asset: {t_asset.status()} | Chunks: {len(t_chunks)} tiles')
#
# # 3. Construir mosaico del país (después de que las tareas hayan concluido)
# dest = assemble_country_mosaic(year, month, 'monthly')
print('Celda programática lista. Descomente el bloque de arriba para usarla.')

## 🧬 M2 — Gestor de Muestras

In [ ]:
from M2_sample_manager import run_ui as sample_ui
sample_interface = sample_ui()

In [ ]:
# ── Uso programático
# from M2_sample_manager import load_sample_fc, filter_samples, get_sample_stats
#
# fc = load_sample_fc('samples_peru_v1')
# fc = filter_samples(fc, version='v1', regions=['peru_r1_amazonia'],
#                     period='monthly', years=[2023, 2024])
# stats = get_sample_stats(fc)
# selected_bands = ['red', 'nir', 'swir1', 'swir2']  # predeterminado
# print(stats)

# ── Recuperar la selección de la interfaz de usuario
try:
    sample_fc, selected_bands = sample_interface.get_selection()
    print(f'✅ {sample_fc.size().getInfo():,} muestras | Bandas: {selected_bands}')
except ValueError as e:
    print(f'⚠️  {e}  — Use la interfaz de usuario de arriba o defina sample_fc y selected_bands manualmente.')

## 🧠 M3 — Entrenamiento del Modelo

In [ ]:
from M3_model_trainer import run_ui as trainer_ui
trainer_interface = trainer_ui(
    sample_fc=sample_fc,
    selected_bands=selected_bands
)

In [ ]:
# ── Uso programático (sin interfaz de usuario) — ideal para la automatización local + scripts por lotes
# from M2_sample_manager import samples_to_array
# from M3_model_trainer import ModelTrainer
# from M0_auth_config import CONFIG
#
# bands = ['red', 'nir', 'swir1', 'swir2']
# X, y  = samples_to_array(sample_fc, bands, CONFIG['asset_mosaics'])
#
# trainer = ModelTrainer(num_input=len(bands))
# trainer._bands_input  = bands
# trainer._sample_count = {'burned': int(y.sum()), 'not_burned': int((y==0).sum())}
# trainer.train(X, y)
# hp = trainer.save(version='v1', region='peru_r1')
# print('Modelo guardado:', hp['gcs_model_path'])
print('Celda programática lista.')

## 🔥 M4 — Clasificación

In [ ]:
from M4_classifier import run_ui as classifier_ui
classifier_interface = classifier_ui()

In [ ]:
# ── Uso programático (lote)
# from M4_classifier import run_classification_campaign
#
# results = run_classification_campaign(
#     year    = 2024,
#     months  = [7, 8, 9, 10],
#     regions = ['peru_r1_amazonia', 'peru_r2_sierra'],
#     version = 'v1'
# )
# for r in results:
#     print(f"{r['year']}-{r['month']:02d}: {r['tiles_done']}/{r['tiles_total']} fragmentos")
print('Celda programática lista.')

## 📢 M5 — Publicación

In [ ]:
from M5_publisher import run_ui as publisher_ui
publisher_interface = publisher_ui()

In [ ]:
# ── Uso programático
# from M5_publisher import assemble_classified_mosaic, publish_to_gee
# import ee
#
# year, month = 2024, 8
# regions     = ['peru_r1_amazonia']
# version     = 'v1'
#
# # 1. Construir COG del mosaico de clasificación
# dest = assemble_classified_mosaic(year, month, regions, version, draft=True)
#
# # 2. Cargar de GCS como imagen de EE
# classified = ee.Image.loadGeoTIFF(dest)
#
# # 3. Aplicar máscara LULC + publicar
# task, asset_id, desc = publish_to_gee(
#     classified, year, month, regions, version, is_final=False
# )
# print(f'Asset: {asset_id}')
print('Celda programática lista.')

---
## 🗂️ Estructura de Archivos Generados

```
GCS: gs://mapbiomas-fire/
└── sudamerica/peru/monitor/
    ├── library_images/
    │   ├── monthly/
    │   │   ├── chunks/{year}/{month:02d}/    ← COG por mosaico (mosaico)
    │   │   └── mosaics/{year}/{month:02d}/   ← COG país (mosaico)
    │   └── yearly/
    │       ├── chunks/{year}/
    │       └── mosaics/{year}/
    ├── classifications/
    │   └── monthly/{year}/{month:02d}/
    │       ├── *_cls.tif                     ← mosaicos clasificados
    │       └── mosaics/*_cog.tif             ← mosaico final
    └── models/{version}/{region}/
        ├── weights.npz
        └── hyperparameters.json

GEE Assets: projects/mapbiomas-mosaics/assets/SENTINEL/FIRE/mosaics-countries/
            projects/mapbiomas-peru/assets/FIRE/MONITOR/classification/
```